In [5]:
import pandas as pd
import numpy as np
import joblib

# Load data
cbb_26 = pd.read_csv('Real2026Data.csv')

round_map_feat = {
    "R64": 1,
    "R32": 2,
    "S16": 3,
    "E8": 4,
    "F4": 5,
    "2ND": 6,
    "Champions": 7
}

cbb_26["Last_yr_performance"] = (
    cbb_26["Last_yr_performance"]
    .map(round_map_feat)
    .fillna(0)
    .astype(int)
)

cbb_26_need = [
    'ADJOE','ADJDE','BARTHAG','EFG_O','EFG_D','TOR','TORD','ORB',
    'FTR','FTRD','2P_O','2P_D','3P_O','3P_D','ADJ_T',
    'Num_of_champs','Last_yr_performance','Coach_champs'
]

# Keep full dataset with team names
cbb_2026 = cbb_26.copy()
cbb_2026 = cbb_2026[cbb_2026['SEED'] != 'N/A'].copy()

# Features only for prediction
X = cbb_2026[cbb_26_need]

In [6]:
# Load model
model = joblib.load("NCAABballPredictor.pk1")

# Predict probabilities
preds = model.predict_proba(X)

# Compute expected round score
expected_round = (preds * np.array([1,2,3,4,5,6,7])).sum(axis=1)

# Attach scores to teams
cbb_2026['score'] = expected_round

# Sort by best teams
cbb_2026 = cbb_2026.sort_values('score', ascending=False).reset_index(drop=True)

# Assign bracket outcomes
round_assignments = (
    ["CHAMPION"] * 1 +
    ["2ND"] * 1 +
    ["F4"] * 2 +
    ["E8"] * 4 +
    ["R16"] * 8 +
    ["R32"] * 16 +
    ["R64"] * 32
)

cbb_2026 = cbb_2026.head(64)

cbb_2026['Final_Prediction'] = round_assignments

# Show results with team names
with pd.option_context('display.max_rows', None, 'display.max_columns', None):
    print(cbb_2026[['TEAM', 'SEED', 'score', 'Final_Prediction']])



                TEAM  SEED     score Final_Prediction
0            Florida     1  4.378428         CHAMPION
1            Arizona     1  4.015709              2ND
2               Duke     1  3.963730               F4
3           Michigan     1  3.963219               F4
4            Houston     2  3.880558               E8
5        Connecticut     2  3.828837               E8
6           Illinois     3  3.773014               E8
7             Purdue     2  3.743446               E8
8           Arkansas     4  3.735813              R16
9           Iowa St.     2  3.730568              R16
10      Michigan St.     3  3.629565              R16
11        Louisville     6  3.621759              R16
12           Alabama     4  3.570728              R16
13        Texas Tech     5  3.557981              R16
14          Virginia     3  3.557387              R16
15         Wisconsin     5  3.538071              R16
16           Gonzaga     3  3.506587              R32
17        St. John's     5  